In [1]:
import json
from pathlib import Path

In [2]:
ROOT = Path("/mnt/c/FYLAB/Data Projects Department/PostOil")
INPUT_PATH = ROOT / "images" / "custom.geo.json"
OUTPUT_PATH = ROOT / "images" / "gcc.geo.json"
GCC_NAMES = {
    "Bahrain": "bahrain",
    "Kuwait": "kuwait",
    "Oman": "oman",
    "Qatar": "qatar",
    "Saudi Arabia": "saudi-arabia",
    "United Arab Emirates": "united-arab-emirates",
}

In [3]:
def get_name(props: dict) -> str | None:
    # Priorite aux champs usuels
    for k in ("name", "NAME", "admin", "ADMIN"):
        v = props.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None
def get_iso_a3(props: dict) -> str | None:
    for k in ("iso_a3", "ISO_A3", "adm0_a3", "ADM0_A3"):
        v = props.get(k)
        if isinstance(v, str) and v.strip():
            return v.strip()
    return None
def main():
    with INPUT_PATH.open("r", encoding="utf-8") as f:
        geo = json.load(f)
    if geo.get("type") != "FeatureCollection":
        raise ValueError("Input is not a FeatureCollection")
    features = geo.get("features", [])
    out_features = []
    found = set()
    for ft in features:
        props = ft.get("properties") or {}
        name = get_name(props)
        if not name or name not in GCC_NAMES:
            continue
        found.add(name)
        # Propriétés minimales pour le frontend
        new_props = {
            "name": name,
            "slug": GCC_NAMES[name],
            "iso_a3": get_iso_a3(props),
        }
        out_features.append({
            "type": "Feature",
            "properties": new_props,
            "geometry": ft.get("geometry"),
        })
    missing = set(GCC_NAMES.keys()) - found
    if missing:
        raise RuntimeError(f"Missing GCC countries in source: {sorted(missing)}")
    out_geo = {
        "type": "FeatureCollection",
        "features": out_features,
    }
    with OUTPUT_PATH.open("w", encoding="utf-8") as f:
        json.dump(out_geo, f, ensure_ascii=False, separators=(",", ":"))
    print(f"Saved {len(out_features)} features to: {OUTPUT_PATH}")
if __name__ == "__main__":
    main()

Saved 6 features to: /mnt/c/FYLAB/Data Projects Department/PostOil/images/gcc.geo.json
